In [ ]:
import os
import h5py
import torch
from torch.utils.data import Dataset
import numpy as np
import torch
from torch import optim
import torch.nn as nn
from torch.utils.data import DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from numpy import random
import cv2
from numpy import identity

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
print(os.listdir('/content/drive/MyDrive'))

In [ ]:
!unzip -q "/content/drive/MyDrive/Nepal_SR.zip" -d /content/
# !unzip -q "/content/drive/MyDrive/Validation.zip" -d /content/

In [ ]:
print(os.listdir('/content'))

In [ ]:
def compute_topographical_features(dem, slope, res=10.0):
    """ Compute northness, eastness, profile curvature"""

    dem_padded = np.pad(dem, pad_width=1, mode="edge")

    dy, dx = np.gradient(dem_padded, res)

    d2y, _ = np.gradient(dy, res)
    _, d2x = np.gradient(dx, res)

    # removing the padding added in the beginning
    dx = dx[1:-1, 1:-1]
    dy = dy[1:-1, 1:-1]
    d2x = d2x[1:-1, 1:-1]
    d2y = d2y[1:-1, 1:-1]

    aspect = np.arctan2(-dy, dx)
    northness = np.cos(aspect)
    eastness = np.sin(aspect)

    curvature = d2x + d2y

    return northness, eastness, curvature


def compute_normalization(img_dir, file_ids): 
    """ Compute the mean and the standard deviation from the training set only. Protected against NaN vlaues. """
    N_CHANNELS = 17
    channel_sum = np.zeros(N_CHANNELS, dtype=np.float64)
    channel_squared_sum = np.zeros(N_CHANNELS, dtype=np.float64)
    pixel_count = 0
    eps = 1e-6
    
    for file_id in file_ids:
        img_path = os.path.join(img_dir, f"image_{file_id}.h5")
        if not os.path.exists(img_path):
            continue  # Skip if the file does not exist

        with h5py.File(img_path, "r") as f:
            raw_image = f["img"][:]
            
        blue  = raw_image[:, :, 1].astype(np.float32)
        green = raw_image[:, :, 2].astype(np.float32)
        red   = raw_image[:, :, 3].astype(np.float32)
        b5    = raw_image[:, :, 4].astype(np.float32)
        b6    = raw_image[:, :, 5].astype(np.float32)
        b7    = raw_image[:, :, 6].astype(np.float32)
        nir   = raw_image[:, :, 7].astype(np.float32)
        swir1 = raw_image[:, :, 10].astype(np.float32)
        swir2 = raw_image[:, :, 11].astype(np.float32)
        slope = raw_image[:, :, 12].astype(np.float32)
        dem   = raw_image[:, :, 13].astype(np.float32)
        
        northness, eastness, curvature = compute_topographical_features(dem, slope)
        
        ndvi = (nir - red) / (nir + red + eps)
        bsi = ((swir1 + red) - (nir + blue)) / ((swir1 + red) + (nir + blue) + eps)
        ndwi = (green - nir) / (green + nir + eps)
        
        image_17ch = np.stack(
            [dem, slope, northness, eastness, curvature, blue, green, red, nir, b5, b6, b7, swir1, swir2, ndvi, bsi, ndwi], axis=-1
        )  # final 17 channel raster
        
        image_17ch = np.nan_to_num(image_17ch, nan=0.0)  # Replace NaN values with 0.0
        
        h, w, _ = image_17ch.shape
        
        channel_sum += np.sum(image_17ch, axis=(0, 1))
        channel_squared_sum += np.sum(image_17ch ** 2, axis=(0, 1))
        pixel_count += h * w

    means = channel_sum / pixel_count
    stds = np.sqrt((channel_squared_sum / pixel_count) - (means ** 2))
    
    return means.astype(np.float32), stds.astype(np.float32)

def train_transform(means, stds):
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),

        A.ShiftScaleRotate(
            shift_limit=0.05,
            scale_limit=(-0.1, 0.1),
            rotate_limit=10,
            border_mode=cv2.BORDER_REFLECT,
            p=0.5
        ),
        # Applying normalization here
        # it does img = (img - mean * max_pixel_value) / (std * max_pixel_value) inder the hood
        A.Normalize(mean=list(means), std=list(stds), max_pixel_value=1.0),

        ToTensorV2()
    ])


def val_transform(means, stds):
    return A.Compose([
        A.Normalize(mean=list(means), std=list(stds), max_pixel_value=1.0),
        ToTensorV2()
    ])

class LandslideDataset(Dataset):

    def __init__(self, img_dir, mask_dir=None, transform=None, file_ids = None):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.transform = transform

        if file_ids is not None:
            self.file_ids = file_ids
        else:
            self.file_ids = sorted(
                [
                    int(f.split("_")[1].split(".")[0])
                    for f in os.listdir(img_dir)
                    if f.endswith(".h5")
                ]
            )

    def __len__(self):
        return len(self.file_ids)

    def __getitem__(self, idx):
        file_id = self.file_ids[idx]
        img_name = f"image_{file_id}.h5"
        mask_name = f"mask_{file_id}.h5"

        with h5py.File(os.path.join(self.img_dir, img_name), "r") as f:
            raw_image = f["img"][:]

        if self.mask_dir is not None:
            with h5py.File(os.path.join(self.mask_dir, mask_name), "r") as f:
                mask = f["mask"][:]
        else:
            mask = np.zeros((128, 128), dtype=np.int64)

        eps = 1e-6

        blue = raw_image[:, :, 1]
        green = raw_image[:, :, 2]
        red = raw_image[:, :, 3]
        b5 = raw_image[:, :, 4]
        b6 = raw_image[:, :, 5]
        b7 = raw_image[:, :, 6]
        nir = raw_image[:, :, 7]
        swir1 = raw_image[:, :, 10]
        swir2 = raw_image[:, :, 11]
        
        # Terrain features
        slope = raw_image[:, :, 12]
        dem = raw_image[:, :, 13]

        northness, eastness, curvature = compute_topographical_features(dem, slope)

        ndvi = (nir - red) / (nir + red + eps)

        bsi = ((swir1 + red) - (nir + blue)) / ((swir1 + red) + (nir + blue) + eps)

        ndwi = (green - nir) / (green + nir + eps)

        # axis=-1 means the new axis is added at the END → shape: (128, 128, 17)
        image_17ch = np.stack(
            [dem, slope, northness, eastness, curvature, blue, green, red, nir, b5, b6, b7, swir1, swir2, ndvi, bsi, ndwi], axis=-1
        ).astype(np.float32)  # Final shape: (128, 128, 17)

        image_17ch = np.nan_to_num(image_17ch, nan=0.0, posinf=0.0, neginf=0.0)

    # this will do the normalization, rotation and convert to the tensors
        if self.transform:

            augmented = self.transform(image=image_17ch, mask=mask)
            image = augmented['image'].float()
            mask = augmented['mask'].long()
        else:
            image_17ch = image_17ch.transpose((2, 0, 1))  # (C, H, W)

            image = torch.from_numpy(image_17ch).float()
            mask = torch.from_numpy(mask).long()
            

        return image, mask


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# --------------ResUNet Model----------------------

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        
        # First convolution
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=False)
        
        # Second convolution
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # Shortcut connection
        self.shortcut = nn.Sequential()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        identity = self.shortcut(x)
        
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        
        out = self.conv2(out)
        out = self.bn2(out)
        
        out = out + identity  # Out-of-place addition
        out = self.relu(out)
        
        return out


# ---- Encoder Block ---- #

class EncoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = ResidualBlock(in_channels, out_channels)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        features = self.conv(x)
        pooled = self.pool(features)
        return features, pooled


# ---- Decoder Block ---- #

class DecoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.upsample = nn.ConvTranspose2d(
            in_channels, out_channels, kernel_size=2, stride=2
        )
        self.conv = ResidualBlock(out_channels * 2, out_channels)

    def forward(self, x , skip):
            upsampled = self.upsample(x)
            cat = torch.cat([upsampled, skip], dim=1)
            x = self.conv(cat)
            return x


class ResUNet(nn.Module):
    def __init__(self, in_channels=17, num_classes=2):
        super().__init__()

        # -- Encoding Phase -- 
        self.enc1 = EncoderBlock(in_channels, 64)
        self.enc2 = EncoderBlock(64, 128)
        self.enc3 = EncoderBlock(128, 256)
        self.enc4 = EncoderBlock(256, 512)

        # -- Bottleneck (deepest point - no pooling here) --
        self.bottleneck = ResidualBlock(512, 1024)

        # -- Decoding Phase --
        self.dec4 = DecoderBlock(1024, 512)
        self.dec3 = DecoderBlock(512, 256)
        self.dec2 = DecoderBlock(256, 128)
        self.dec1 = DecoderBlock(128, 64)

        # -- Final Output Layer -- 
        self.output_conv = nn.Conv2d(64, num_classes, kernel_size=1)

    def forward(self, x):
        # --- Encoder ---
        skip1, x = self.enc1(x)
        skip2, x = self.enc2(x)
        skip3, x = self.enc3(x)
        skip4, x = self.enc4(x)

        # --- Bottleneck ---
        x = self.bottleneck(x)

        # --- Decoder ---
        x = self.dec4(x, skip4)
        x = self.dec3(x, skip3)
        x = self.dec2(x, skip2)
        x = self.dec1(x, skip1)

        # Final output
        return self.output_conv(x)

In [ ]:
import torch
import torch.nn as nn


class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
 
    def forward(self, predictions, targets):
        probs = torch.softmax(predictions, dim=1)[:, 1, :, :] 
        targets_f = targets.float()
 
        intersection = (probs * targets_f).sum(dim=(1, 2))
 
        dice = (2.0 * intersection + self.smooth) / (
            probs.sum(dim=(1, 2)) + targets_f.sum(dim=(1, 2)) + self.smooth
        )
        return 1 - dice.mean()


class BinaryFocalLoss(nn.Module):
    """Focal Loss for binary segmentation (landslide vs background).
       Down-weights easy background pixels and focuses on hard, rare landslide pixels."""
    def __init__(self, alpha, gamma, smooth=1e-6):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.smooth = smooth
        
    def forward(self, logits, targets):
        probs = torch.softmax(logits, dim=1)[:, 1]      # probability of landslide
        targets_f = targets.float()
        
        pt = torch.where(targets_f == 1, probs, 1 - probs)
        
        alpha_t = torch.where(targets_f == 1, self.alpha, 1 - self.alpha)
        
        focal = -alpha_t * (1 - pt) ** self.gamma * torch.log(pt + self.smooth)
        
        return focal.mean()


class CombinedFocalDiceLoss(nn.Module):
    """Blend of Focal Loss (handles class imbalance) and Dice Loss (maximises region overlap).
       Often outperforms CE+Dice for extremely imbalanced tasks like landslide detection."""
    def __init__(self, focal_weight, dice_weight, alpha, gamma):
        super().__init__()
        self.focal = BinaryFocalLoss(alpha=alpha, gamma=gamma)
        self.dice = DiceLoss()
        self.focal_weight = focal_weight
        self.dice_weight = dice_weight

    def forward(self, predictions, targets):
        return (self.focal_weight * self.focal(predictions, targets) +
                self.dice_weight * self.dice(predictions, targets))


In [ ]:
def compute_metrics(predictions, targets, threshold=0.5):
    """
    predictions : (Batch, 2, H, W) raw logits from the model
    targets     : (Batch, H, W)    ground truth integer labels
    threshold   : float probability threshold for converting landslide probability to binary
    """
    # Convert logits to landslide probability (channel 1) and apply threshold
    probs = torch.softmax(predictions, dim=1)[:, 1]   # shape: (Batch, H, W)
    pred_bin = probs > threshold


    # True Positives, False Positives, False Negatives (with thresholded preds)
    tp = ((targets == 1) & (pred_bin == 1)).sum().float() 
    fp = ((targets == 0) & (pred_bin == 1)).sum().float()
    fn = ((targets == 1) & (pred_bin == 0)).sum().float()

    return tp, fp, fn

In [ ]:
# import os
# import torch
# import random
# import torch.optim as optim
# from torch.utils.data import DataLoader


# def train_transfer_learning(
#     train_img_dir,
#     train_mask_dir, 
#     val_img_dir,
#     val_mask_dir,
#     img_dir,
#     mask_dir,
#     phase1_epochs,
#     phase2_epochs,
#     batch_size, 
#     pretrained_model_path,
#     save_path
# ):
#     device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#     print(f"Using {device} for the trainng!!!")
    
#     train_files = sorted([f for f in os.listdir(train_img_dir) if f.endswith(".h5")])
#     train_ids = [int(f.split("_")[1].split(".")[0]) for f in train_files]
    
    
#     val_files = sorted([f for f in os.listdir(val_img_dir) if f.endswith(".h5")])
#     all_val_ids = [int(f.split("_")[1].split(".")[0]) for f in val_files]

#     random.seed(42)
#     random.shuffle(all_val_ids)
    
#     split_idx = len(all_val_ids) // 2
#     val_ids = all_val_ids[:split_idx]
#     test_ids = all_val_ids[split_idx:]
    
#     print(f"Train: {len(train_ids)} | Val: {len(val_ids)} | Test: {len(test_ids)}")
    
#     print("\n ---- Computing Normalization Statistics From Nepal Training Split ----")
#     MEANS, STDS = compute_normalization(train_img_dir, train_ids)
#     print(f"Computed Means: {MEANS}")
#     print(f"Computed Stds: {STDS}\n")
    
#     # Initializing Datasets
#     train_dataset = LandslideDataset(train_img_dir, train_mask_dir, transform=train_transform(MEANS, STDS), file_ids=train_ids)
#     val_dataset = LandslideDataset(val_img_dir, val_mask_dir, transform=val_transform(MEANS, STDS), file_ids=val_ids)    
#     test_dataset =  LandslideDataset(val_img_dir, val_mask_dir, transform=val_transform(MEANS, STDS), file_ids=test_ids)
        
#      # Initializing Dataloaders
#     train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)   
#     val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)   
#     test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)   
    
#     # -------------- MODEL, LOSS & PRETRAINED WEIGHTS ----------------- #
                        
#     model = ResUNet(in_channels=17, num_classes=2).to(device)
#     criterion = CombinedFocalDiceLoss(focal_weight=0.35, dice_weight=0.65, alpha=0.50, gamma=2.0)
    
#     print(f" Loading pretrained weights from {pretrained_model_path}....")
#     checkpoint = torch.load(pretrained_model_path, map_location=device)
#     pretrained_dict = checkpoint['model_state_dict']
    
#     # ------------ Loading the pretrained model ------------ #
#     model_dict = model.state_dict()
#     pretrained_dict = {k: v for k, v in pretrained_dict.items() if k in model_dict and v.size() == model_dict[k].size()}
#     model_dict.update(pretrained_dict)
#     model.load_state_dict(model_dict)
#     print(f" [INFO] Loaded {len(pretrained_dict)} matching layer dictionaries.")
    
#     best_val_f1 = 0.0 
    
#     def run_epoch(epoch, total_epochs, optimizer, scheduler, phase_name):
#         nonlocal best_val_f1
        
#         model.train()
        
#         # AVOIDING THE BATCH NORMALIZATION TRAP
#         if phase_name == "Phase 1":
#             for name, module in model.named_modules():
#                 if 'enc' in name or 'bottleneck' in name:
#                     module.eval()
                    

#         running_train_loss = 0.0
#         for images, targets in train_loader:
#             images, targets = images.to(device), targets.to(device)
#             optimizer.zero_grad()
#             predictions = model(images)
#             loss = criterion(predictions, targets)
#             loss.backward()
#             optimizer.step()
#             running_train_loss = running_train_loss + loss.item()
            
#         train_loss = running_train_loss / len(train_loader)
        
        
#         # Validation
#         model.eval()
#         running_val_loss = 0.0
#         total_tp, total_fp, total_fn = 0, 0, 0
        
#         with torch.no_grad():
#             for images, targets in val_loader:
#                 images, targets = images.to(device), targets.to(device)
#                 predictions = model(images)
#                 loss = criterion(predictions, targets)
#                 running_val_loss = running_val_loss + loss.item()
                
#                 batch_metrics = compute_metrics(predictions, targets)
#                 total_tp += batch_metrics[0]
#                 total_fp += batch_metrics[1]
#                 total_fn += batch_metrics[2]
                
#         val_loss = running_val_loss / len(val_loader)
#         iou = total_tp / (total_tp + total_fp + total_fn + 1e-6)
#         f1 = 2 * total_tp / (2 * total_tp + total_fp + total_fn + 1e-6)
        
#         scheduler.step(1 - iou)
        
#         print(
#             f"[{phase_name}] Epoch [{epoch:02d}/{total_epochs}] "
#             f"| Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val IoU: {iou:.4f} | Val F1: {f1:.4f} "
#             f"| LR: {optimizer.param_groups[0]['lr']:.6f}"            
#         )
        
#         # --- Save Checkpoint ---
#         if f1 > best_val_f1:
#             best_val_f1 = f1
#             torch.save({
#                 'model_state_dict': model.state_dict(),
#                 'optimizer_state_dict': optimizer.state_dict(),
#                 'best_val_f1': best_val_f1
#             }, save_path)
#             print(f" => Saved new best model checkpoint! F1: {best_val_f1:.4f}")
        
                    
#     # ========================================
#     # PHASE 1: FREEZE ENCODER & TRAIN DECODER
#     # ========================================
#     print("\n" + "="*55)
#     print("PHASE 1: Feature Extraction (Encoder Frozen) ")
#     print("="*55)
    
#     # Freeze Encoder leyer
#     for name, param in model.named_parameters():
#         if 'enc' in name or 'bottleneck' in name:
#             param.requires_grad = False
#         else:
#             param.requires_grad = True # Decoder and output remain active
            
#     optimizer_p1 = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4, weight_decay=1e-4)
#     scheduler_p1 = optim.lr_scheduler.ReduceLROnPlateau(optimizer_p1, mode="min", patience=3, factor=0.5)
    
#     best_phase1_iou = 0.0
#     epochs_without_improvement = 0
    
#     for epoch in range(1, phase1_epochs + 1):
#         metrics = run_epoch(epoch, phase1_epochs, optimizer_p1, scheduler_p1, "Phase 1")
        
#         if metrics["iou"] > best_phase1_iou:
#             best_phase1_iou = metrics["iou"]
#             epochs_without_improvement = 0
#         else:
#             epochs_without_improvement += 1
            

        
#     # ========================================
#     # PHASE 2: UNFREEZE ALL AND FINE TUNING
#     # ========================================
    
#     print("\n" + "="*55)
#     print(" PHASE 2: Full Fine-Tuning (All Layers Unfrozen) ")
#     print("="*55)
    
#     # Unfreeze all the layer
    
#     for param in model.parameters():
#         param.requires_grad = True
        
#     # Much Lower learining rate
#     optimizer_p2 = optim.Adam(model.parameters(), lr=1e-5, weight_decay=1e-4)
#     scheduler_p2 = optim.lr_scheduler.ReduceLROnPlateau(optimizer_p2, mode="min", patience=5, factor=0.5)
    
#     for epoch in range(1, phase2_epochs + 1):
#         run_epoch(epoch, phase2_epochs, optimizer_p2, scheduler_p2, "Phase 2")
        
#     # ==========================================
#     # FINAL PHASE: UNBIASED TEST EVALUATION
#     # ==========================================
#     print("\n" + "="*55)
#     print(" EVALUATING BEST MODEL ON UNSEEN TEST SET ")
#     print("="*55)
    
#     # loading the best weights found during training
#     best_checkpoint = torch.load(save_path)
#     model.load_state_dict(best_checkpoint['model_state_dict'])
#     model.eval()
    
#     test_tp, test_fp, test_fn = 0, 0, 0
#     with torch.no_grad():
#         for images, targets in test_loader:
#             images, targets = images.to(device), targets.to(device)
#             predictions = model(images)
#             batch_metrics = compute_metrics(predictions, targets)
#             test_tp += batch_metrics[0]
#             test_fp += batch_metrics[1]
#             test_fn += batch_metrics[2]

#     test_iou = test_tp / (test_tp + test_fp + test_fn + 1e-6)
#     test_f1 = 2 * test_tp / (2 * test_tp + test_fp + test_fn + 1e-6)
#     test_precision = test_tp / (test_tp + test_fp + 1e-6)
#     test_recall = test_tp / (test_tp + test_fn + 1e-6)

#     print(f"Final Test Metrics -> IoU: {test_iou:.4f} | F1: {test_f1:.4f} | Precision: {test_precision:.4f} | Recall: {test_recall:.4f}\n")
#     print(f"[SUCCESS] Transfer Learning Pipeline Complete. Model saved to: {save_path}")
    
#     return model

In [ ]:
# TRAIN_IMG_DIR  = "/content/Nepal_SR/img"
# TRAIN_MASK_DIR = "/content/Nepal_SR/masks"
# VAL_IMG_DIR    = "/content/Validation/img"
# VAL_MASK_DIR   = "/content/Validation/masks"

# IMG_DIR  = "/content/Nepal_SR/img"
# MASK_DIR = "/content/Nepal_SR/masks"

# PRETRAINED_MODEL = "/content/drive/MyDrive/landslide4sense_model.pth"
# NEW_SAVE_PATH    = "/content/nepal_transfer_learning_model.pth" 

# trained_model = train_transfer_learning(
#     img_dir    = IMG_DIR,
#     mask_dir   = MASK_DIR,
#     train_img_dir         = TRAIN_IMG_DIR,
#     train_mask_dir        = TRAIN_MASK_DIR,
#     val_img_dir           = VAL_IMG_DIR,
#     val_mask_dir          = VAL_MASK_DIR,
#     phase1_epochs         = 15,
#     phase2_epochs         = 60, 
#     batch_size            = 16,
#     pretrained_model_path = PRETRAINED_MODEL,
#     save_path             = NEW_SAVE_PATH
# )

In [ ]:
import os
import random
import numpy as np
import torch
import torch.optim as optim
from torch.utils.data import DataLoader


def set_seed(seed):
    """FIxes every source of randomness: Pythons random, numpy used for the random flips, rotations, and torch/cuda (weight init for unfrozen layers, dropout, etc.)"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    
def get_all_ids(img_dir):
    all_files = sorted([f for f in os.listdir(img_dir) if f.endswith(".h5")])
    # tile_42.h5, it first splits the _ and takes [1] and then splits the . and takes 42
    return [int(f.split("_")[1].split(".")[0]) for f in all_files]

def load_pretrained_backbone(model, checkpoint_path, device):
    print(f"Loading the pretrained weights from {checkpoint_path} ...")
    # device: regardless of what it was trained on, load on whatever the device is present
    checkpoint = torch.load(checkpoint_path, map_location=device)
    
    pretrained_dict = checkpoint["model_state_dict"] if "model_state_dict" in checkpoint else checkpoint
    
    model_dict = model.state_dict()
    
    # if the k (like enc1.conv1.weight) and v(the values of weight) inside matches then copy to the matched and update the dict
    matched = {k: v for k, v in pretrained_dict.items() if k in model_dict and v.size() == model_dict[k].size()}
    skipped = [k for k in pretrained_dict if k not in matched]
    unfilled = [k for k in model_dict if k not in matched]
    
    # update the matched dict with the dummy dict
    model_dict.update(matched)
    model.load_state_dict(model_dict)
    
    print(f"[INFO] Loaded {len(matched)} / {len(model_dict)} matching layer tensors.")
    if skipped:
        print(f"[WARNING] {len(skipped)} checkpoint tensors did NOT match (name/shape) ")
        
    if unfilled:
            print(f"[WARNING] {len(unfilled)} checkpoint tensors had no matching checkpoint entry ")
    
    if not skipped and not unfilled:
        print(f"[INFO] Every tensor matched exactly - clean full load successful.")
        
    return model
        
    
DECODER_KEYS = ["dec4", "dec3", "dec2", "dec1", "output_conv"]
ENC4_BOTTLENECK_KEYS = ["enc4", "bottleneck"]
ENC123_KEYS = ["enc1", "enc2", "enc3"]

def get_module_dict(model):
    return {
        "enc1": model.enc1, "enc2": model.enc2, "enc3": model.enc3,
        "enc4": model.enc4, "bottleneck": model.bottleneck,
        "dec4": model.dec4, "dec3": model.dec3, "dec2": model.dec2, 
        "dec1": model.dec1, "output_conv": model.output_conv,
    }
    

def set_requires_grad(modules_dict, trainable_keys):
    all_keys = set(modules_dict.keys()) # extracts all the layer name from the models dict
    trainable_keys = set(trainable_keys) # converts the trainable keys to set
    for key in all_keys:
        req = key in trainable_keys # evaluates to a bollen value if presesnt true else false (req is either True or False)
        for p in modules_dict[key].parameters(): # loops through the weight matrix and bias vector (p)
            p.requires_grad = req # if req is True requires_grad = True and viceversa
            
            
# This scoops up the exact weight tensors for the given modules_dict
def params_for_keys(modules_dict, keys):
    params_list = []
    
    for key in keys: # loop over each module name
        module = modules_dict[key] # grab the actual python module(eg: model.dec4)
        for p in module.parameters(): # loop over every weight and bias matrix and append to list
            params_list.append(p)
            
    return params_list

    
def set_training_mode_with_frozen(model, modules_dict, frozen_keys):
    """ model.train() puts every submodule including the ones meant to be frozen to train, so we explicitly force the frozen modules back to .eval() afterward """
    model.train()
    for key in frozen_keys:
        modules_dict[key].eval()
        
def train_transfer_learning(
    img_dir,
    mask_dir,
    pretrained_model_path,
    save_path, 
    val_split=0.15,
    phase0_max_epoch=15,
    phase0_patience=4,
    phase1_max_epoch=30,
    phase1_patience=6,
    phase2_max_epoch=80,
    phase2_patience=20,
    batch_size=16,
    seed=42, 
):
    set_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    all_ids = get_all_ids(img_dir)
    random.shuffle(all_ids)
    val_size = int(val_split * len(all_ids))
    val_ids = all_ids[:val_size]
    train_ids = all_ids[val_size:]
    print(f"Total: {len(all_ids)} | Train: {len(train_ids)} | Val: {len(val_ids)}")
    
    print("------ Computing Normalization stats from training split ------")
    MEANS, STDS = compute_normalization(img_dir, train_ids)
    print(f"Means:  {MEANS}")
    print(f"Stds:   {STDS}\n")
    
    train_dataset = LandslideDataset(img_dir=img_dir, mask_dir=mask_dir, transform=train_transform(MEANS, STDS), file_ids=train_ids)
    
    val_dataset = LandslideDataset(img_dir=img_dir, mask_dir=mask_dir, transform=val_transform(MEANS, STDS), file_ids=val_ids)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    
    # ----------- Model + pretrained weights ---------------- #
    model = ResUNet(in_channels=17, num_classes=2).to(device)
    model = load_pretrained_backbone(model, pretrained_model_path, device)
    modules_dict = get_module_dict(model)
    
    criterion = CombinedFocalDiceLoss(focal_weight=0.35, dice_weight=0.65, alpha=0.65, gamma=2.0)
    best_val_iou = 0.0
    
    def run_epoch(epoch, total_epochs, optimizer, scheduler, phase_name, frozen_keys):
        nonlocal best_val_iou
        
        set_training_mode_with_frozen(model, modules_dict, frozen_keys)
        running_train_loss = 0.0
        for images, targets in train_loader:
            images, targets = images.to(device), targets.to(device)
            optimizer.zero_grad() # clears out the stored gradients from previous batch
            predictions = model(images) # forward pass (feeds the input image to the resunet model)
            loss = criterion(predictions, targets) # calculates the batch loss using the combinedfocaldiceloss
            loss.backward()
            optimizer.step()
            running_train_loss += loss.item()
        train_loss = running_train_loss / len(train_loader)
        
        model.eval()
        running_val_loss = 0.0
        total_tp, total_fp, total_fn = 0, 0, 0
        with torch.no_grad():
            for images, targets in val_loader:
                images, targets = images.to(device), targets.to(device)
                predictions = model(images)
                loss = criterion(predictions, targets)
                running_val_loss += loss.item()
                tp, fp, fn = compute_metrics(predictions, targets)
                total_tp += tp
                total_fp += fp
                total_fn += fn
        val_loss = running_val_loss / len(val_loader)
        
        iou = total_tp / (total_tp + total_fp + total_fn + 1e-6)
        f1 = 2 * total_tp / (2 * total_tp + total_fp + total_fn + 1e-6)
        precision = total_tp / (total_tp + total_fp + 1e-6)
        recall = total_tp / (total_tp + total_fn + 1e-6)
        
        scheduler.step(1 - iou)
        
        lrs = ", ".join(f"{g['lr']:.2e}" for g in optimizer.param_groups)
        print(
            f"[{phase_name}] Epoch [{epoch:03d}/{total_epochs}] "
            f"| Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} "
            f"| IoU: {iou:.4f} F1: {f1:.4f} Precision: {precision:.4f} Recall: {recall:.4f} "
            f"| LR(s): {lrs}"
        )
        
        if iou > best_val_iou:
            best_val_iou = iou
            torch.save({
                "phase": phase_name,
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "val_iou": iou, 
                "val_f1": f1,
                "val_precision": precision,
                "val_recall": recall,
            }, save_path)
            print(f" => Saved new best checkpoint (IoU={best_val_iou:.4f}, phase:{phase_name}) -> {save_path}")
            
        return {"iou": iou, "f1": f1, "train_loss": train_loss, "val_loss": val_loss}
        
    def run_phase(phase_name, max_epochs, patience, optimizer, frozen_keys):
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", patience=max(1, patience // 2), # sets the schedulers patience to half of the phase's early stopping patience
            factor=0.5 # halves the learning rate whenever the iou  plateaus
        )
        best_phase_iou = 0.0
        epochs_without_improvement = 0
        
        for epoch in range(1, max_epochs + 1):
            metrics = run_epoch(epoch, max_epochs, optimizer, scheduler, phase_name, frozen_keys)
            
            if metrics["iou"] > best_phase_iou:
                best_phase_iou = metrics["iou"]
                epochs_without_improvement = 0
            else:
                epochs_without_improvement += 1
                
            if epochs_without_improvement >= patience:
                print(f"{phase_name} early stopping at epoch {epoch}" 
                      f"(no improvement for {patience} epochs ..)")
                return
                
        print(f"{phase_name} reached its safety cap of {max_epochs} epochs without early stopping...")
        
    
    # ======================================================================
    # Phase 0: WARMUP - decoder + output head only, everything else frozen
    # ======================================================================
    print("\n" + "=" * 60)
    print("PHASE 0: Warmup (Decoder + Output Head only)")
    print("\n" + "=" * 60)
    
    
    set_requires_grad(modules_dict, trainable_keys=DECODER_KEYS)
    decoder_params = params_for_keys(modules_dict, DECODER_KEYS)
    optimizer_p0 = optim.Adam([{"params": decoder_params, "lr": 1e-3}], weight_decay=1e-4)
    
    run_phase("Phase 0", phase0_max_epoch, phase0_patience, optimizer_p0, frozen_keys=ENC123_KEYS + ENC4_BOTTLENECK_KEYS)
    
    # ======================================================================
    # Phase 1: TOP ENCODER - unfreeze bottleneck + enc4, enc123 stay frozen
    # ======================================================================
    
    print("\n" + "=" * 60)
    print("PHASE 1: Top Encoder (Decoder + Bottleneck + enc4 trainable)")
    print("\n" + "=" * 60)
    
    set_requires_grad(modules_dict, trainable_keys=ENC4_BOTTLENECK_KEYS + DECODER_KEYS)
    decoder_params = params_for_keys(modules_dict, DECODER_KEYS)
    enc4_bottleneck_param = params_for_keys(modules_dict, ENC4_BOTTLENECK_KEYS)
    
    optimizer_p1 = optim.Adam([
        {"params": enc4_bottleneck_param, "lr": 1e-4},
        {"params": decoder_params, "lr": 5e-4},
    ], weight_decay=1e-4)
    
    run_phase("Phase 1", phase1_max_epoch, phase1_patience, optimizer_p1, frozen_keys=ENC123_KEYS)
    
    # ========================================================================
    # Phase 2: Everything unfrozen
    # ========================================================================
    
    print("\n" + "=" * 60)
    print("PHASE 2: Everything Unfrozen")
    print("\n" + "=" * 60)
    
    set_requires_grad(modules_dict, trainable_keys=ENC123_KEYS + ENC4_BOTTLENECK_KEYS + DECODER_KEYS)
    decoder_params = params_for_keys(modules_dict, DECODER_KEYS)
    encoder_params = params_for_keys(modules_dict, ENC123_KEYS + ENC4_BOTTLENECK_KEYS)
    
    optimizer_p2 = optim.Adam([
        {"params": encoder_params, "lr": 1e-5},
        {"params": decoder_params, "lr": 1e-4},
    ], weight_decay=1e-4)
    
    run_phase("Phase 2", phase2_max_epoch, phase2_patience, optimizer_p2, frozen_keys=[])
    
    print(f"\n Training Complete. Best IoU accross all phases: {best_val_iou:.4f}")
    print(f"Best checkpoint saved to: {save_path}")
    return model

In [ ]:
if __name__ == "__main__":
    train_transfer_learning(
        img_dir="/content/Nepal_SR/img",
        mask_dir="/content/Nepal_SR/masks",
        pretrained_model_path= "/content/drive/MyDrive/landslide4sense_model.pth",
        save_path="/content/nepal_transfer_learning_model.pth",
        val_split=0.15,
        phase0_max_epoch=15,
        phase0_patience=4,
        phase1_max_epoch=30,
        phase1_patience=6,
        phase2_max_epoch=80,
        phase2_patience=20,
        batch_size=16,
        seed=42,
    )

In [ ]:
print(os.listdir('/content/drive/MyDrive'))
